# FASE 1: Exploración y diagnóstico del Dataset

In [8]:
import pandas as pd
# Leemos el archivo CSV.
df = pd.read_csv('../dataset/healthcare_dataset.csv')

# Mostramos información
df.head(10)

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
5,EMILY JOHNSOn,36,Male,A+,Asthma,2023-12-20,Taylor Newton,Nunez-Humphrey,UnitedHealthcare,48145.110951,389,Urgent,2023-12-24,Ibuprofen,Normal
6,edwArD EDWaRDs,21,Female,AB-,Diabetes,2020-11-03,Kelly Olson,Group Middleton,Medicare,19580.872345,389,Emergency,2020-11-15,Paracetamol,Inconclusive
7,CHrisTInA MARtinez,20,Female,A+,Cancer,2021-12-28,Suzanne Thomas,"Powell Robinson and Valdez,",Cigna,45820.462722,277,Emergency,2022-01-07,Paracetamol,Inconclusive
8,JASmINe aGuIlaR,82,Male,AB+,Asthma,2020-07-01,Daniel Ferguson,Sons Rich and,Cigna,50119.222792,316,Elective,2020-07-14,Aspirin,Abnormal
9,ChRISTopher BerG,58,Female,AB-,Cancer,2021-05-23,Heather Day,Padilla-Walker,UnitedHealthcare,19784.631062,249,Elective,2021-06-22,Paracetamol,Inconclusive


- Existe un error ya que los nombres se encuentran combinados entre mayúsculas y minúsculas, es decir, no se encuentran normalizados. Además que algunos datos tienen abreviaturas de sus profesiones.

- Los valores de "Billing Amount" tienen muchos decimales.

- Los hospitales tiene terminaciones o inicios con la palabra "and" lo que provoca confusión ya que parece que falta una parte.

In [2]:
# Mostraremos si existen valores nulos por cada columna y qué tipo de dato es cada una de ellas
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55499 entries, 0 to 55498
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55499 non-null  object 
 1   Age                 55499 non-null  int64  
 2   Gender              55499 non-null  object 
 3   Blood Type          55499 non-null  object 
 4   Medical Condition   55499 non-null  object 
 5   Date of Admission   55499 non-null  object 
 6   Doctor              55499 non-null  object 
 7   Hospital            55499 non-null  object 
 8   Insurance Provider  55499 non-null  object 
 9   Billing Amount      55499 non-null  float64
 10  Room Number         55499 non-null  int64  
 11  Admission Type      55499 non-null  object 
 12  Discharge Date      55499 non-null  object 
 13  Medication          55499 non-null  object 
 14  Test Results        55499 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

- Las fechas de admisión y de alta medica son de tipo object, deberían ser de tipo date para poder trabajar estos datos posteriormente.

In [3]:
# Mostraremos un resumen estadístico de las columnas numéricas
df.describe()

,Age,Billing Amount,Room Number
count,55499.000000,55499.000000,55499.000000
mean,51.539541,25539.585242,301.138309
std,19.602621,14211.441017,115.241191
min,13.000000,-2008.492140,101.000000
25%,35.000000,13242.214898,202.000000
50%,52.000000,25538.524706,302.000000
75%,68.000000,37820.527579,401.000000
max,89.000000,52764.276736,500.000000


- En la columna "Billing amount" tiene valores negativos, lo cual genera error porque no pueden existir valores negativos es en columna.

In [4]:
# Buscamos el error, si los pacientes que fueron dados de alta antes de haber sido ingresados
# Convertimos a formato a fecha 
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'])

# Examinamos los casos ilógicos
fechas_ilógicas = df[df['Discharge Date'] < df['Date of Admission']]

print(f"Encontramos {len(fechas_ilógicas)} registros en el que se produce un caso ilógico.")
fechas_ilógicas[['Name', 'Date of Admission', 'Discharge Date']].head()

Encontramos 0 registros en el que se produce un caso ilógico.


,Name,Date of Admission,Discharge Date


In [ ]:
# Buscamos si existen datos duplicados.
# Columnas que definen un ingreso único
columnas_identidad = ['Name', 'Age', 'Blood Type', 'Date of Admission', 'Doctor', 'Hospital']

# Encontrar quiénes están repetidos y cuántas veces
reporte_duplicados = df.groupby(columnas_identidad).size().reset_index(name='Cantidad')

# Filtrar solo los que aparecen más de una vez
solo_repetidos = reporte_duplicados[reporte_duplicados['Cantidad'] > 1].sort_values(by='Cantidad', ascending=False)

if not solo_repetidos.empty:
    print(f"Se encontraron {len(solo_repetidos)} casos de ingresos duplicados.")
    print("\nResumen de los casos más repetidos:")
    print(solo_repetidos.head()) 
else:
    print("No se encontraron ingresos duplicados.")

Se encontraron 534 casos de ingresos duplicados.

Resumen de los casos más repetidos:
                    Name  Age Blood Type Date of Admission           Doctor  \
50         ABIgaIL YOung   41         O+        2022-12-15    Edward Kramer   
37587           ivan LEe   35        AB+        2021-09-20      Stacey Lamb   
38930   jOSEPH scHRoeDeR   43         O+        2022-04-21      Dylan Scott   
38619  jEsSIcA cONtrEras   25         A-        2022-07-24      Gina Garcia   
38492     jEnNIfeR PaTel   62         A-        2021-11-07  Kenneth Shepard   

             Hospital  Cantidad  
50     Moore-Mcdaniel         2  
37587  Castillo-Walsh         2  
38930       Ltd Patel         2  
38619     Evans-Kelly         2  
38492     Lopez-Allen         2  


- Existen problemas de duplicidad de datos.

In [6]:
# Buscamos datos que tengan todo el registro identico pero varia solo en la columna de fecha
# Usamos .nunique() para ver cuántas edades distintas hay por cada grupo
conteo_edades = df.groupby(['Name', 'Date of Admission'])['Age'].nunique().reset_index(name='Variaciones_Edad')

# Filtramos los casos donde hay más de una edad para el mismo ingreso
casos_conflicto = conteo_edades[conteo_edades['Variaciones_Edad'] > 1]

# Mostramos los registros específicos para comparar
pacientes_con_error = casos_conflicto['Name'].unique()
print(len(pacientes_con_error))
print(df[df['Name'].isin(pacientes_con_error)].sort_values(by=['Name', 'Date of Admission']))


4965
                   Name  Age  Gender Blood Type Medical Condition  \
17082       AARon smITh   79    Male         A-            Cancer   
54392       AARon smITh   81    Male         A-            Cancer   
40304    ABIGAiL wateRS   34  Female         O+            Asthma   
52110    ABIGAiL wateRS   35  Female         O+            Asthma   
2018     ABIgAIL tucKeR   66    Male         B+            Cancer   
...                 ...  ...     ...        ...               ...   
53939    zacHary fLOrEs   63    Male        AB-            Cancer   
15564     zachAry Brown   69  Female         A+           Obesity   
50502     zachAry Brown   72  Female         A+           Obesity   
1902   zacharY BauTista   43  Female        AB+            Cancer   
50726  zacharY BauTista   46  Female        AB+            Cancer   

      Date of Admission             Doctor                   Hospital  \
17082        2019-11-21        Gina Jacobs               Weber-Warren   
54392        2019-11

- Se puede observar que existen datos con inconsistencia lógica, ya que tienen todas las columnas identicas excepto la columna de la edad.

In [7]:
# Filtramos los registros donde el año de admisión sea mayor a 2026
fechas_futuras = df[df['Date of Admission'].dt.year > 2026]

print(f"Se encontraron {len(fechas_futuras)} registros con fechas del futuro.")

# Si el número es mayor a 0, mostramos cuáles son para investigar
if len(fechas_futuras) > 0:
    print("\nDetalle de los registros del futuro:")
    # display() se ve más bonito en Jupyter que print() para los DataFrames
    display(fechas_futuras[['Name', 'Date of Admission', 'Discharge Date', 'Hospital']].head())

Se encontraron 0 registros con fechas del futuro.
